#### Exploration du feature engineering

Ce notebook teste des transformations candidates avant leur implémentation
définitive dans `src/bank_marketing/domain/preprocessing.py`.

Les données brutes ne sont jamais modifiées. Les transformations sont réalisées
sur une copie afin de pouvoir être reproduites et validées par l'équipe.

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

# Définir l'emplacement du jeu de données brut
DATA_PATH = Path("../../data/raw/bank-additional.csv")

# Charger les données sans modifier le fichier source
df_raw = pd.read_csv(DATA_PATH, sep=";")

# Créer une copie de travail dédiée aux transformations
df_features = df_raw.copy()

# Vérifier les dimensions initiales
df_features.shape

(4119, 21)

#### Transformation de `pdays`

La valeur `999` signifie qu'un client n'a jamais été contacté lors d'une
campagne précédente. Cette valeur ne représente donc pas réellement un nombre
de jours et ne doit pas être utilisée telle quelle par le modèle.

Deux variables candidates seront créées :

- `previously_contacted` : indique si le client a déjà été contacté ;
- `days_since_previous_contact` : conserve le nombre de jours uniquement pour
  les clients ayant déjà été contactés.

In [7]:
# Créer un indicateur : 1 si le client a déjà été contacté, 0 sinon
df_features["previously_contacted"] = (
    df_features["pdays"].ne(999).astype("int8")
)

# Remplacer 999 par une valeur manquante dans la variable de délai réelle
df_features["days_since_previous_contact"] = df_features["pdays"].replace(
    999,
    np.nan,
)

# Vérifier le résultat de la transformation
df_features[
    [
        "pdays",
        "previously_contacted",
        "days_since_previous_contact",
        "previous",
        "poutcome",
    ]
].head(10)

,pdays,previously_contacted,days_since_previous_contact,previous,poutcome
0,999,0,NaN,0,nonexistent
1,999,0,NaN,0,nonexistent
2,999,0,NaN,0,nonexistent
3,999,0,NaN,0,nonexistent
4,999,0,NaN,0,nonexistent
5,999,0,NaN,2,failure
6,999,0,NaN,0,nonexistent
7,999,0,NaN,0,nonexistent
8,999,0,NaN,1,failure
9,999,0,NaN,0,nonexistent


In [8]:
# Contrôler la cohérence et la répartition des nouvelles variables
pd.DataFrame(
    {
        "nombre_clients": [
            len(df_features),
            df_features["previously_contacted"].sum(),
            df_features["days_since_previous_contact"].notna().sum(),
        ]
    },
    index=[
        "Total",
        "Déjà contactés",
        "Délai de contact renseigné",
    ],
)

,nombre_clients
Total,4119
Déjà contactés,160
Délai de contact renseigné,160


#### Décision provisoire sur `pdays`

Seuls 160 clients sur 4 119 ont déjà été contactés lors d'une campagne
précédente, soit environ 3,88 % des observations.

La valeur `999` ne représente pas un délai réel. La variable `pdays` sera donc
retirée des variables brutes utilisées par le modèle. Elle est remplacée par :

- `previously_contacted`, un indicateur de contact antérieur ;
- `days_since_previous_contact`, renseignée uniquement lorsqu'un contact
  antérieur existe.

L'utilité de `days_since_previous_contact` sera évaluée durant la modélisation.

In [9]:
# Lister les variables qui ne doivent pas être utilisées directement par le modèle
columns_to_exclude = [
    "y",          # La cible est séparée des variables explicatives
    "duration",   # Information connue uniquement après l'appel téléphonique
    "pdays",      # Remplacée par des variables plus interprétables
]

# Construire les variables explicatives candidates
X_candidate = df_features.drop(columns=columns_to_exclude)

# Construire la cible binaire : 1 pour une souscription, 0 sinon
y_target = df_features["y"].eq("yes").astype("int8")

# Vérifier que duration et pdays sont bien absentes des variables candidates
print("Variables candidates :", X_candidate.shape[1])
print("duration présente :", "duration" in X_candidate.columns)
print("pdays présente :", "pdays" in X_candidate.columns)
print("Taux de souscription :", f"{y_target.mean():.2%}")

Variables candidates : 20
duration présente : False
pdays présente : False
Taux de souscription : 10.95%


#### Analyse des modalités `unknown`

Les valeurs `unknown` ne sont pas des valeurs manquantes classiques : elles
signifient que l'information n'a pas été renseignée ou n'est pas disponible.

Avant toute transformation, leur effet sur la souscription est comparé à celui
des clients pour lesquels l'information est connue. À ce stade, aucune modalité
`unknown` n'est remplacée arbitrairement.

In [10]:
# Sélectionner les variables catégorielles pouvant contenir la modalité unknown
categorical_columns = df_features.select_dtypes(include="object").columns.drop("y")

unknown_results = []

for column in categorical_columns:
    # Identifier les lignes dont la modalité est unknown
    is_unknown = df_features[column].eq("unknown")

    # Calculer les informations uniquement si unknown existe dans la variable
    if is_unknown.any():
        unknown_results.append(
            {
                "variable": column,
                "nombre_unknown": int(is_unknown.sum()),
                "pourcentage_unknown": round(is_unknown.mean() * 100, 2),
                "taux_souscription_unknown_pct": round(
                    y_target[is_unknown].mean() * 100,
                    2,
                ),
                "taux_souscription_connu_pct": round(
                    y_target[~is_unknown].mean() * 100,
                    2,
                ),
            }
        )

# Afficher les variables contenant effectivement des modalités unknown
unknown_impact_table = (
    pd.DataFrame(unknown_results)
    .sort_values("pourcentage_unknown", ascending=False)
    .reset_index(drop=True)
)

unknown_impact_table

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_12356\1853835027.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df_features.select_dtypes(include="object").columns.drop("y")


,variable,nombre_unknown,pourcentage_unknown,taux_souscription_unknown_pct,taux_souscription_connu_pct
0,default,803,19.50,6.10,12.12
1,education,167,4.05,15.57,10.75
2,housing,105,2.55,8.57,11.01
3,loan,105,2.55,8.57,11.01
4,job,39,0.95,10.26,10.96
5,marital,11,0.27,9.09,10.95


#### Décision provisoire sur les modalités `unknown`

Les modalités `unknown` sont conservées comme catégories distinctes.

Elles peuvent contenir une information utile : par exemple, les clients dont la
variable `default` est égale à `unknown` ont un taux de souscription de 6,10 %,
contre 12,12 % lorsque cette information est connue.

Aucune modalité `unknown` ne sera donc remplacée manuellement par `no`, `single`
ou toute autre catégorie. Elles seront encodées avec les autres modalités lors
du prétraitement.

In [12]:
# Définir une première liste de variables numériques candidates
numeric_features_candidate = [
    "age",
    "campaign",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
    "previously_contacted",
    "days_since_previous_contact",
]

# Définir les variables catégorielles candidates
categorical_features_candidate = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

# Regrouper toutes les variables actuellement candidates
all_features_candidate = (
    numeric_features_candidate + categorical_features_candidate
)

# Vérifier que toutes les variables prévues existent bien
missing_features = set(all_features_candidate) - set(X_candidate.columns)

print("Nombre de variables candidates :", len(all_features_candidate))
print("Variables absentes :", missing_features)

Nombre de variables candidates : 20
Variables absentes : set()


In [13]:
# Jeu complet : toutes les variables candidates identifiées précédemment
features_complete = all_features_candidate.copy()

# Jeu interprétable : moins de redondance et variables jugées plus pertinentes
features_interpretable = [
    # Variables numériques
    "age",
    "campaign",
    "previous",
    "euribor3m",
    "previously_contacted",

    # Variables catégorielles
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome",
]

# Regrouper les configurations pour les réutiliser plus tard
feature_sets = {
    "complet": features_complete,
    "interpretable": features_interpretable,
}

# Résumer les deux jeux de variables
feature_sets_summary = pd.DataFrame(
    {
        "jeu_variables": list(feature_sets.keys()),
        "nombre_variables": [
            len(features) for features in feature_sets.values()
        ],
        "variables": [
            ", ".join(features) for features in feature_sets.values()
        ],
    }
)

feature_sets_summary

,jeu_variables,nombre_variables,variables
0,complet,20,"age, campaign, previous, emp.var.rate, cons.pr..."
1,interpretable,14,"age, campaign, previous, euribor3m, previously..."


#### Vérification des variables candidates

Avant la modélisation, les jeux de variables doivent être contrôlés :

- aucune fuite d'information ne doit être présente ;
- les valeurs manquantes créées par les transformations doivent être identifiées ;
- le traitement des valeurs manquantes sera effectué plus tard dans le pipeline,
  en étant appris uniquement sur les données d'entraînement.

In [14]:
# Vérifier les valeurs manquantes dans le jeu complet
missing_features_table = pd.DataFrame(
    {
        "nombre_manquant": X_candidate[features_complete].isna().sum(),
        "pourcentage_manquant": (
            X_candidate[features_complete].isna().mean() * 100
        ).round(2),
    }
).sort_values("nombre_manquant", ascending=False)

missing_features_table

,nombre_manquant,pourcentage_manquant
days_since_previous_contact,3959,96.12
age,0,0.00
previous,0,0.00
campaign,0,0.00
emp.var.rate,0,0.00
cons.price.idx,0,0.00
euribor3m,0,0.00
cons.conf.idx,0,0.00
nr.employed,0,0.00
previously_contacted,0,0.00


In [15]:
# Définir les variables interdites pour éviter les fuites d'information
forbidden_features = {
    "y",
    "duration",
    "pdays",
}

feature_validation_results = []

for feature_set_name, features in feature_sets.items():
    forbidden_found = sorted(set(features) & forbidden_features)

    feature_validation_results.append(
        {
            "jeu_variables": feature_set_name,
            "nombre_variables": len(features),
            "variables_interdites": (
                ", ".join(forbidden_found)
                if forbidden_found
                else "Aucune"
            ),
        }
    )

pd.DataFrame(feature_validation_results)

,jeu_variables,nombre_variables,variables_interdites
0,complet,20,Aucune
1,interpretable,14,Aucune


### Décision sur `days_since_previous_contact`

La variable `days_since_previous_contact` est renseignée pour seulement 3,88 %
des clients. Elle est donc retirée des variables candidates afin d'éviter une
imputation massive et une complexité inutile.

L'historique des campagnes précédentes reste représenté par `previously_contacted`,
`previous` et `poutcome`.

In [16]:
# Définir les variables numériques retenues pour la modélisation
numeric_features_candidate = [
    "age",
    "campaign",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
    "previously_contacted",
]

# Reconstruire la liste complète des variables candidates
all_features_candidate = (
    numeric_features_candidate + categorical_features_candidate
)

# Mettre à jour le jeu complet avec cette nouvelle décision
features_complete = all_features_candidate.copy()

# Mettre à jour les configurations de variables
feature_sets["complet"] = features_complete

# Vérifier le nombre de variables de chaque jeu
pd.DataFrame(
    {
        "jeu_variables": list(feature_sets.keys()),
        "nombre_variables": [
            len(features) for features in feature_sets.values()
        ],
    }
)

,jeu_variables,nombre_variables
0,complet,19
1,interpretable,14


In [17]:
# Construire les deux matrices de variables qui seront utilisées plus tard
X_complete = df_features[features_complete].copy()
X_interpretable = df_features[features_interpretable].copy()

# Vérifier leurs dimensions et l'absence de valeurs manquantes
feature_engineering_check = pd.DataFrame(
    {
        "jeu_variables": ["complet", "interpretable"],
        "lignes": [X_complete.shape[0], X_interpretable.shape[0]],
        "colonnes": [X_complete.shape[1], X_interpretable.shape[1]],
        "valeurs_manquantes": [
            int(X_complete.isna().sum().sum()),
            int(X_interpretable.isna().sum().sum()),
        ],
    }
)

feature_engineering_check

,jeu_variables,lignes,colonnes,valeurs_manquantes
0,complet,4119,19,0
1,interpretable,4119,14,0


## Conclusion du feature engineering exploratoire

Les transformations suivantes ont été retenues :

- `duration` est exclue, car elle est connue après l'appel téléphonique ;
- `pdays` est retirée, car la valeur `999` ne correspond pas à un délai réel ;
- `previously_contacted` est créée pour indiquer un contact lors d'une campagne
  antérieure ;
- `days_since_previous_contact` est retirée, car elle serait manquante pour
  96,12 % des clients ;
- les modalités `unknown` sont conservées comme catégories ;
- deux jeux de variables seront comparés : `complet` et `interpretable`.

Les transformations finales devront être implémentées dans le code source,
puis appliquées avant le prétraitement et la modélisation.